---
## 1. Imports & Configuration

In [7]:
import os
import math
import random
from collections import Counter, defaultdict

import pyarrow.parquet as pq
from tqdm import tqdm

# ── Paths ──────────────────────────────────────────────────────────────────
PARQUET_DIR = os.path.join("..", "Assignment_1", "data", "output")
FIRST_SHARD = os.path.join(PARQUET_DIR, "indiccorp_te_tokenized_part-00001.parquet")

# ── Parameters ─────────────────────────────────────────────────────────────
NUM_SENTENCES = 1_000_000
DEV_SIZE      = 1_000
TEST_SIZE     = 1_000
TRAIN_SIZE    = NUM_SENTENCES - DEV_SIZE - TEST_SIZE   # 998,000
SEED          = 42

# Special tokens for sentence boundaries
SOS = "<s>"
EOS = "</s>"
UNK = "<UNK>"

# N-gram orders
ORDERS = (1, 2, 3, 4)
ORDER_NAMES = {1: "Unigram", 2: "Bigram", 3: "Trigram", 4: "Quadgram"}

print(f"Will load {NUM_SENTENCES:,} sentences from:")
print(f"  {os.path.abspath(FIRST_SHARD)}")
print(f"Split: train={TRAIN_SIZE:,}  dev={DEV_SIZE:,}  test={TEST_SIZE:,}")

Will load 1,000,000 sentences from:
  d:\Assignments_sem_5\NLP\Assignment_1\data\output\indiccorp_te_tokenized_part-00001.parquet
Split: train=998,000  dev=1,000  test=1,000


---
## 2. Load Data & Train / Dev / Test Split

Same loading and splitting procedure as Assignment 4 (same seed = 42) for consistency.

In [8]:
# Read the first parquet shard
table = pq.read_table(FIRST_SHARD, columns=["tokenized_sentence"])
raw_sentences = table.column("tokenized_sentence").to_pylist()

# Tokenize
sentences = [s.split() for s in raw_sentences[:NUM_SENTENCES]]

# Shuffle with same seed as Assignment 4 for identical splits
random.seed(SEED)
random.shuffle(sentences)

train_sentences = sentences[:TRAIN_SIZE]
dev_sentences   = sentences[TRAIN_SIZE : TRAIN_SIZE + DEV_SIZE]
test_sentences  = sentences[TRAIN_SIZE + DEV_SIZE : TRAIN_SIZE + DEV_SIZE + TEST_SIZE]

print(f"Training set   : {len(train_sentences):>10,} sentences")
print(f"Development set: {len(dev_sentences):>10,} sentences")
print(f"Test set       : {len(test_sentences):>10,} sentences")

del raw_sentences, table, sentences

Training set   :    998,000 sentences
Development set:      1,000 sentences
Test set       :      1,000 sentences


---
## 3. Build Vocabulary & Utility Functions

In [9]:
# ── Build vocabulary from training data only ─────────────────────────────
vocab = set()
for sent in tqdm(train_sentences, desc="Building vocabulary"):
    vocab.update(sent)

vocab.add(SOS)
vocab.add(EOS)
vocab.add(UNK)

V = len(vocab)
print(f"Vocabulary size (V): {V:,}")

Building vocabulary: 100%|██████████| 998000/998000 [00:04<00:00, 240416.11it/s]

Vocabulary size (V): 670,753


In [10]:
def normalize_token(token):
    """Map out-of-vocabulary tokens to <UNK>."""
    return token if token in vocab else UNK


def pad_sentence(tokens, n):
    """
    Pad a sentence with (n-1) <s> tokens at the start and one </s> at the end.
    Also maps OOV tokens to <UNK>.
    """
    normalized = [normalize_token(t) for t in tokens]
    return [SOS] * (n - 1) + normalized + [EOS]


def get_ngrams(tokens, n):
    """Extract all n-grams from a token list."""
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


# Quick demo
demo = ["నమస్తే", "తెలంగాణ", "ప్రత్యేక"]
for n in [1, 2, 3, 4]:
    padded = pad_sentence(demo, n)
    ngrams = get_ngrams(padded, n)
    print(f"n={n}: padded={padded}")
    print(f"       n-grams={ngrams}\n")

n=1: padded=['నమస్తే', 'తెలంగాణ', 'ప్రత్యేక', '</s>']
       n-grams=[('నమస్తే',), ('తెలంగాణ',), ('ప్రత్యేక',), ('</s>',)]

n=2: padded=['<s>', 'నమస్తే', 'తెలంగాణ', 'ప్రత్యేక', '</s>']
       n-grams=[('<s>', 'నమస్తే'), ('నమస్తే', 'తెలంగాణ'), ('తెలంగాణ', 'ప్రత్యేక'), ('ప్రత్యేక', '</s>')]

n=3: padded=['<s>', '<s>', 'నమస్తే', 'తెలంగాణ', 'ప్రత్యేక', '</s>']
       n-grams=[('<s>', '<s>', 'నమస్తే'), ('<s>', 'నమస్తే', 'తెలంగాణ'), ('నమస్తే', 'తెలంగాణ', 'ప్రత్యేక'), ('తెలంగాణ', 'ప్రత్యేక', '</s>')]

n=4: padded=['<s>', '<s>', '<s>', 'నమస్తే', 'తెలంగాణ', 'ప్రత్యేక', '</s>']
       n-grams=[('<s>', '<s>', '<s>', 'నమస్తే'), ('<s>', '<s>', 'నమస్తే', 'తెలంగాణ'), ('<s>', 'నమస్తే', 'తెలంగాణ', 'ప్రత్యేక'), ('నమస్తే', 'తెలంగాణ', 'ప్రత్యేక', '</s>')]



---
## 4. Build N-gram Counts from Training Data

We build:
- **`counts[n]`**: Counter of n-gram tuples → frequency
- **`contexts[n]`**: Counter of (n-1)-gram context tuples → frequency (for n ≥ 2)
- **`followers[n]`**: context → set of unique words that follow the context
- **`predecessors[n]`**: word → set of unique contexts that precede the word

In [ ]:
# ── Count n-grams for n = 1, 2, 3, 4 ─────────────────────────────────────
counts       = {n: Counter() for n in ORDERS}
contexts     = {n: Counter() for n in (2, 3, 4)}
followers    = {n: defaultdict(set) for n in ORDERS}     # context → {words that follow}
predecessors = {n: defaultdict(set) for n in ORDERS}     # word → {contexts that precede}

for sent in tqdm(train_sentences, desc="Counting n-grams"):
    for n in ORDERS:
        padded = pad_sentence(sent, n)
        for gram in get_ngrams(padded, n):
            counts[n][gram] += 1
            if n > 1:
                contexts[n][gram[:-1]] += 1
            # Track follower and predecessor sets (needed for Kneser-Ney)
            followers[n][gram[:-1]].add(gram[-1])
            predecessors[n][gram[-1]].add(gram[:-1])

N_total = sum(counts[1].values())   # total unigram tokens

print(f"\n{'Model':<12} {'Unique n-grams':>16} {'Total count':>14}")
print("-" * 46)
for n in ORDERS:
    print(f"{ORDER_NAMES[n]:<12} {len(counts[n]):>16,} {sum(counts[n].values()):>14,}")

Counting n-grams:  98%|█████████▊| 980212/998000 [12:52<06:30, 45.51it/s]   

---
## 5. MLE (Baseline — No Smoothing)

Maximum Likelihood Estimation probabilities. These will produce **infinite perplexity** whenever
an unseen n-gram appears in the dev/test data, but they serve as the foundation for smoothing.

In [ ]:
def mle_prob(order, history, word):
    """
    MLE probability: P(word | history) = C(history, word) / C(history)
    For unigrams: P(word) = C(word) / N
    """
    word = normalize_token(word)
    if order == 1:
        return counts[1].get((word,), 0) / N_total
    history = tuple(normalize_token(t) for t in history)
    denominator = contexts[order].get(history, 0)
    if denominator == 0:
        return 0.0
    numerator = counts[order].get(history + (word,), 0)
    return numerator / denominator


# Quick test
print(f"P_MLE(</s> | .) = {mle_prob(2, ('.', ), EOS):.6f}")
print(f"P_MLE(ఈ | <s>) = {mle_prob(2, (SOS,), 'ఈ'):.6f}")

---
## 6. Perplexity Computation Helper

A generic function that computes perplexity given a probability function and n-gram order.

In [ ]:
def safe_log2(p):
    """Safe log2 that avoids log(0) by clamping to a very small value."""
    return math.log2(max(p, 1e-300))


def sentence_log_prob(sentence, prob_fn, order):
    """
    Compute log2 probability of a sentence under a given probability function.
    
    Parameters
    ----------
    sentence : list of str
        Raw token list.
    prob_fn : callable(order, history_tuple, word) -> float
        The smoothed probability function.
    order : int
        The n-gram order (1, 2, 3, or 4).
    
    Returns
    -------
    (log2_prob, n_predicted_tokens)
    """
    padded = pad_sentence(sentence, order)
    log_prob = 0.0
    n_tokens = len(padded) - order + 1   # number of predicted tokens
    
    for i in range(order - 1, len(padded)):
        history = tuple(padded[i - order + 1 : i])
        word = padded[i]
        p = prob_fn(order, history, word)
        log_prob += safe_log2(p)
    
    return log_prob, n_tokens


def compute_perplexity(data_sentences, prob_fn, order):
    """
    Compute perplexity of a dataset:
        PP = 2^{-1/N * Σ log2 P(w_i | context)}
    """
    total_log_prob = 0.0
    total_tokens   = 0
    
    for sent in data_sentences:
        lp, nt = sentence_log_prob(sent, prob_fn, order)
        total_log_prob += lp
        total_tokens   += nt
    
    avg_log_prob = total_log_prob / total_tokens
    return 2 ** (-avg_log_prob)


print("Perplexity helper ready.")

---
## 7. Smoothing Technique 1 — Interpolated Smoothing

**Interpolated Smoothing** combines probabilities from multiple n-gram orders using
learned or hand-set weights:

$$P_{\text{interp}}(w_i \mid w_{i-n+1}^{i-1}) = \sum_{k=1}^{n} \lambda_k \cdot P_{\text{MLE}}^{(k)}(w_i \mid w_{i-k+1}^{i-1})$$

where $\sum_k \lambda_k = 1$.

For the **Unigram** model, interpolation is not applicable (only one order), so we use MLE
with a small floor for unseen words.

In [ ]:
# ── Interpolation weights (tuned by hand; sum to 1 for each order) ────────
# Higher-order models get more weight as they capture more context.
INTERP_WEIGHTS = {
    1: (1.0,),                    # Unigram: just MLE
    2: (0.4, 0.6),               # λ₁=0.4 (unigram), λ₂=0.6 (bigram)
    3: (0.2, 0.3, 0.5),          # λ₁=0.2, λ₂=0.3, λ₃=0.5
    4: (0.1, 0.2, 0.3, 0.4),     # λ₁=0.1, λ₂=0.2, λ₃=0.3, λ₄=0.4
}


def interpolated_prob(order, history, word):
    """
    Interpolated smoothing: weighted combination of MLE probabilities
    from order-1 down to unigram.
    
    For unigram (order=1), returns MLE with a floor of 1/N to avoid zero.
    """
    word = normalize_token(word)
    if order == 1:
        return max(mle_prob(1, (), word), 1 / N_total)
    
    weights = INTERP_WEIGHTS[order]
    prob = 0.0
    for k, lam in enumerate(weights, start=1):
        # k=1 → unigram MLE, k=2 → bigram MLE, etc.
        if k == 1:
            sub_history = ()
        else:
            sub_history = tuple(history[-(k-1):])
        prob += lam * mle_prob(k, sub_history, word)
    
    return max(prob, 1e-300)


# ── Demo ──────────────────────────────────────────────────────────────────
print("Interpolated Smoothing Weights:")
for n, w in INTERP_WEIGHTS.items():
    print(f"  {ORDER_NAMES[n]}: λ = {w}")

print(f"\nP_interp(</s> | .) [bigram]  = {interpolated_prob(2, ('.', ), EOS):.6f}")
print(f"P_interp(ఈ | <s>) [bigram]  = {interpolated_prob(2, (SOS,), 'ఈ'):.6f}")
print(f"P_interp(ఈ | <s>,<s>) [tri] = {interpolated_prob(3, (SOS, SOS), 'ఈ'):.6f}")

### 7.1 Evaluate Interpolated Smoothing

In [ ]:
print("Interpolated Smoothing — Perplexity\n")
print(f"{'Model':<12} {'Dev PP':>15} {'Test PP':>15}")
print("-" * 45)

interp_results = {}
for n in ORDERS:
    pp_dev  = compute_perplexity(dev_sentences, interpolated_prob, n)
    pp_test = compute_perplexity(test_sentences, interpolated_prob, n)
    interp_results[n] = {"dev": pp_dev, "test": pp_test}
    print(f"{ORDER_NAMES[n]:<12} {pp_dev:>15,.2f} {pp_test:>15,.2f}")

---
## 8. Smoothing Technique 2 — Good-Turing Smoothing

**Good-Turing** re-estimates the probability of n-grams by adjusting their counts
using the **frequency of frequencies** $N_r$ (how many n-grams occur exactly $r$ times):

$$c^* = (r + 1) \cdot \frac{N_{r+1}}{N_r}$$

For unseen n-grams ($r = 0$): $c^* = N_1 / N_0$

The adjusted counts are then normalized to obtain probabilities.

In [ ]:
def good_turing_adjusted_count(r, freq_of_freq):
    """
    Compute the Good-Turing adjusted count c* for an n-gram with raw count r.
    
    Parameters
    ----------
    r : int
        Raw count of the n-gram.
    freq_of_freq : dict
        Mapping from count r → N_r (number of n-grams occurring exactly r times).
    """
    if r == 0:
        # Unseen n-grams: c* = N_1 / N_0
        N_0 = max(freq_of_freq.get(0, 1), 1)
        N_1 = freq_of_freq.get(1, 0)
        return N_1 / N_0
    
    N_r   = freq_of_freq.get(r, 0)
    N_r1  = freq_of_freq.get(r + 1, 0)
    
    if N_r1 > 0 and N_r > 0:
        return (r + 1) * N_r1 / N_r
    
    # Fallback: if N_{r+1} is not available, keep original count
    return float(r)


# ── Cache for Good-Turing distributions ──────────────────────────────────
gt_cache = {}

def good_turing_distribution(order, history=()):
    """
    Build the full Good-Turing smoothed probability distribution for all
    words in the vocabulary given a context (history).
    
    Returns a dict: word → P_GT(word | history)
    """
    history = tuple(history)
    cache_key = (order, history)
    if cache_key in gt_cache:
        return gt_cache[cache_key]
    
    # Collect raw counts for every word in vocab given the context
    if order == 1:
        observed = {w: counts[1].get((w,), 0) for w in vocab}
    else:
        observed = {w: counts[order].get(history + (w,), 0) for w in vocab}
    
    # Compute frequency-of-frequencies
    freq_of_freq = Counter(c for c in observed.values() if c > 0)
    freq_of_freq[0] = sum(1 for c in observed.values() if c == 0)
    
    # Adjust counts
    adjusted = {
        w: good_turing_adjusted_count(c, freq_of_freq)
        for w, c in observed.items()
    }
    
    # Normalize to probabilities
    total = sum(adjusted.values())
    if total <= 0:
        result = {w: 1 / V for w in vocab}
    else:
        result = {w: val / total for w, val in adjusted.items()}
    
    gt_cache[cache_key] = result
    return result


def good_turing_prob(order, history, word):
    """
    Good-Turing smoothed probability: P_GT(word | history)
    """
    word = normalize_token(word)
    dist = good_turing_distribution(order, history)
    return dist.get(word, 1e-300)


# ── Demo ──────────────────────────────────────────────────────────────────
# Show frequency-of-frequencies for unigrams
unigram_fof = Counter(c for c in counts[1].values())
print("Unigram Frequency-of-Frequencies (top 10):")
for r, Nr in sorted(unigram_fof.items())[:10]:
    print(f"  N_{r} = {Nr:,} (n-grams occurring exactly {r} time{'s' if r != 1 else ''})")

print(f"\nP_GT(</s> | .) [bigram]  = {good_turing_prob(2, ('.', ), EOS):.6f}")
print(f"P_GT(ఈ | <s>) [bigram]  = {good_turing_prob(2, (SOS,), 'ఈ'):.6f}")

### 8.1 Evaluate Good-Turing Smoothing

In [ ]:
print("Good-Turing Smoothing — Perplexity\n")
print(f"{'Model':<12} {'Dev PP':>15} {'Test PP':>15}")
print("-" * 45)

gt_results = {}
for n in ORDERS:
    pp_dev  = compute_perplexity(dev_sentences, good_turing_prob, n)
    pp_test = compute_perplexity(test_sentences, good_turing_prob, n)
    gt_results[n] = {"dev": pp_dev, "test": pp_test}
    print(f"{ORDER_NAMES[n]:<12} {pp_dev:>15,.2f} {pp_test:>15,.2f}")

---
## 9. Smoothing Technique 3 — Katz Back-off Smoothing

**Katz Back-off** uses Good-Turing discounted probabilities for **seen** n-grams,
and **backs off** to a lower-order model for **unseen** n-grams. The back-off
weight $\alpha$ ensures the distribution sums to 1:

$$P_{\text{Katz}}(w_i \mid w_{i-n+1}^{i-1}) = \begin{cases}
\frac{c^*(w_{i-n+1}^i)}{C(w_{i-n+1}^{i-1})} & \text{if } C(w_{i-n+1}^i) > 0 \\
\alpha(w_{i-n+1}^{i-1}) \cdot P_{\text{Katz}}(w_i \mid w_{i-n+2}^{i-1}) & \text{otherwise}
\end{cases}$$

In [ ]:
# ── Precompute frequency-of-frequency tables for each n-gram order ───────
fof_tables = {}
for n in ORDERS:
    fof_tables[n] = Counter(c for c in counts[n].values() if c > 0)
    fof_tables[n][0] = V - len(counts[n])   # approximate N_0

# Cache for Katz back-off alpha values
katz_alpha_cache = {}


def katz_prob(order, history, word):
    """
    Katz Back-off smoothed probability.
    
    - For seen n-grams: use Good-Turing discounted probability.
    - For unseen n-grams: back off to (order-1) model with scaling factor α.
    """
    word = normalize_token(word)
    
    # Base case: unigram → use Good-Turing
    if order == 1:
        return good_turing_prob(1, (), word)
    
    history = tuple(normalize_token(t) for t in history)
    context_count = contexts[order].get(history, 0)
    
    # If context never seen, back off entirely
    if context_count == 0:
        return katz_prob(order - 1, history[1:], word)
    
    # Check if the full n-gram was seen
    ngram_count = counts[order].get(history + (word,), 0)
    
    if ngram_count > 0:
        # Seen n-gram: use Good-Turing adjusted count
        c_star = good_turing_adjusted_count(ngram_count, fof_tables[order])
        return max((c_star / ngram_count) * (ngram_count / context_count), 1e-300)
    
    # Unseen n-gram: back off with alpha scaling
    cache_key = (order, history)
    if cache_key not in katz_alpha_cache:
        seen_words = followers[order].get(history, set())
        
        # Sum of discounted probabilities for seen words
        seen_mass = sum(
            katz_prob(order, history, sw) for sw in seen_words
        )
        
        # Sum of lower-order probabilities for seen words
        lower_mass_for_seen = sum(
            katz_prob(order - 1, history[1:], sw) for sw in seen_words
        )
        
        leftover_mass = max(1.0 - seen_mass, 1e-12)
        lower_leftover = max(1.0 - lower_mass_for_seen, 1e-12)
        
        katz_alpha_cache[cache_key] = leftover_mass / lower_leftover
    
    alpha = katz_alpha_cache[cache_key]
    return alpha * katz_prob(order - 1, history[1:], word)


# ── Demo ──────────────────────────────────────────────────────────────────
print(f"P_Katz(</s> | .) [bigram]  = {katz_prob(2, ('.', ), EOS):.6f}")
print(f"P_Katz(ఈ | <s>) [bigram]  = {katz_prob(2, (SOS,), 'ఈ'):.6f}")

### 9.1 Evaluate Katz Back-off Smoothing

In [ ]:
print("Katz Back-off Smoothing — Perplexity\n")
print(f"{'Model':<12} {'Dev PP':>15} {'Test PP':>15}")
print("-" * 45)

katz_results = {}
for n in ORDERS:
    pp_dev  = compute_perplexity(dev_sentences, katz_prob, n)
    pp_test = compute_perplexity(test_sentences, katz_prob, n)
    katz_results[n] = {"dev": pp_dev, "test": pp_test}
    print(f"{ORDER_NAMES[n]:<12} {pp_dev:>15,.2f} {pp_test:>15,.2f}")

---
## 10. Smoothing Technique 4 — Stupid Back-off

**Stupid Back-off** (Brants et al., 2007) is a simple, non-probabilistic smoothing method.
It does not produce a true probability distribution but assigns **scores**:

$$S(w_i \mid w_{i-n+1}^{i-1}) = \begin{cases}
\frac{C(w_{i-n+1}^i)}{C(w_{i-n+1}^{i-1})} & \text{if } C(w_{i-n+1}^i) > 0 \\
\alpha \cdot S(w_i \mid w_{i-n+2}^{i-1}) & \text{otherwise}
\end{cases}$$

Typically $\alpha = 0.4$. Despite not being a proper distribution, it works
surprisingly well in practice, especially for large corpora.

In [ ]:
STUPID_ALPHA = 0.4   # standard back-off factor


def stupid_backoff_prob(order, history, word):
    """
    Stupid Back-off score.
    
    Not a true probability distribution, but used with perplexity
    as a relative comparison metric.
    """
    word = normalize_token(word)
    
    # Base case: unigram MLE with floor
    if order == 1:
        return max(mle_prob(1, (), word), 1 / N_total)
    
    history = tuple(normalize_token(t) for t in history)
    
    # If the n-gram was seen, return MLE
    if counts[order].get(history + (word,), 0) > 0:
        return max(mle_prob(order, history, word), 1e-300)
    
    # Back off: multiply by α and try lower order
    return STUPID_ALPHA * stupid_backoff_prob(order - 1, history[1:], word)


# ── Demo ──────────────────────────────────────────────────────────────────
print(f"S_stupid(</s> | .) [bigram]  = {stupid_backoff_prob(2, ('.', ), EOS):.6f}")
print(f"S_stupid(ఈ | <s>) [bigram]  = {stupid_backoff_prob(2, (SOS,), 'ఈ'):.6f}")
# Unseen word example
print(f"S_stupid(UNSEEN_XYZ)        = {stupid_backoff_prob(2, (SOS,), 'UNSEEN_XYZ'):.10f}")

### 10.1 Evaluate Stupid Back-off

In [ ]:
print("Stupid Back-off — Perplexity (pseudo-perplexity since scores ≠ probabilities)\n")
print(f"{'Model':<12} {'Dev PP':>15} {'Test PP':>15}")
print("-" * 45)

stupid_results = {}
for n in ORDERS:
    pp_dev  = compute_perplexity(dev_sentences, stupid_backoff_prob, n)
    pp_test = compute_perplexity(test_sentences, stupid_backoff_prob, n)
    stupid_results[n] = {"dev": pp_dev, "test": pp_test}
    print(f"{ORDER_NAMES[n]:<12} {pp_dev:>15,.2f} {pp_test:>15,.2f}")

---
## 11. Smoothing Technique 5 — Kneser-Ney Smoothing

**Kneser-Ney Smoothing** uses **absolute discounting** and replaces the lower-order
distribution with a **continuation probability** — how likely a word is to appear
in novel contexts rather than just how frequently it occurs overall.

$$P_{\text{KN}}(w_i \mid w_{i-n+1}^{i-1}) = 
\frac{\max(C(w_{i-n+1}^i) - d, 0)}{C(w_{i-n+1}^{i-1})}
+ \lambda(w_{i-n+1}^{i-1}) \cdot P_{\text{KN}}(w_i \mid w_{i-n+2}^{i-1})$$

where:
- $d$ is the discount (typically 0.75)
- $\lambda(\text{context}) = \frac{d \cdot |\{w : C(\text{context}, w) > 0\}|}{C(\text{context})}$
- The lowest-order model uses **continuation probability**:
  $P_{\text{cont}}(w) = \frac{|\{v : C(v, w) > 0\}|}{\sum_{w'} |\{v : C(v, w') > 0\}|}$

In [ ]:
KN_DISCOUNT = 0.75

# ── Precompute total unique bigram continuations for P_cont denominator ───
total_unique_bigram_continuations = sum(
    len(ctx_set) for ctx_set in predecessors[2].values()
)
print(f"Total unique bigram continuations: {total_unique_bigram_continuations:,}")


def continuation_prob(word):
    """
    Continuation probability for Kneser-Ney:
    P_cont(word) = |{v : C(v, word) > 0}| / Σ_{w'} |{v : C(v, w') > 0}|
    
    i.e., the number of unique contexts that precede `word` in bigrams,
    divided by the total number of unique bigram type pairings.
    """
    word = normalize_token(word)
    n_predecessors = len(predecessors[2].get(word, set()))
    return n_predecessors / max(total_unique_bigram_continuations, 1)


def kneser_ney_prob(order, history, word):
    """
    Kneser-Ney smoothed probability.
    
    At the lowest order (unigram), uses continuation probability.
    At higher orders, applies absolute discounting and recursive back-off.
    """
    word = normalize_token(word)
    
    # Base case: continuation probability
    if order == 1:
        return max(continuation_prob(word), 1e-300)
    
    history = tuple(normalize_token(t) for t in history)
    context_count = contexts[order].get(history, 0)
    
    # If context never seen, back off entirely
    if context_count == 0:
        return kneser_ney_prob(order - 1, history[1:], word)
    
    # First term: discounted probability
    ngram_count = counts[order].get(history + (word,), 0)
    first_term = max(ngram_count - KN_DISCOUNT, 0.0) / context_count
    
    # Interpolation weight (lambda)
    n_unique_followers = len(followers[order].get(history, set()))
    lam = KN_DISCOUNT * n_unique_followers / context_count
    
    # Recursive lower-order probability
    lower_prob = kneser_ney_prob(order - 1, history[1:], word)
    
    return max(first_term + lam * lower_prob, 1e-300)


# ── Demo ──────────────────────────────────────────────────────────────────
print(f"\nP_KN(</s> | .) [bigram]  = {kneser_ney_prob(2, ('.', ), EOS):.6f}")
print(f"P_KN(ఈ | <s>) [bigram]  = {kneser_ney_prob(2, (SOS,), 'ఈ'):.6f}")
print(f"P_KN(ఈ | <s>,<s>) [tri] = {kneser_ney_prob(3, (SOS, SOS), 'ఈ'):.6f}")
print(f"P_cont(ఈ)              = {continuation_prob('ఈ'):.8f}")
print(f"P_cont(</s>)           = {continuation_prob(EOS):.8f}")

### 11.1 Evaluate Kneser-Ney Smoothing

In [ ]:
print("Kneser-Ney Smoothing — Perplexity\n")
print(f"{'Model':<12} {'Dev PP':>15} {'Test PP':>15}")
print("-" * 45)

kn_results = {}
for n in ORDERS:
    pp_dev  = compute_perplexity(dev_sentences, kneser_ney_prob, n)
    pp_test = compute_perplexity(test_sentences, kneser_ney_prob, n)
    kn_results[n] = {"dev": pp_dev, "test": pp_test}
    print(f"{ORDER_NAMES[n]:<12} {pp_dev:>15,.2f} {pp_test:>15,.2f}")

---
## 12. Comprehensive Comparison — All Smoothing Techniques

Side-by-side perplexity comparison of all 5 smoothing methods across all 4 n-gram models
on both the development and test sets.

In [ ]:
# ── Collect all results ──────────────────────────────────────────────────
all_methods = {
    "Interpolated":  interp_results,
    "Good-Turing":   gt_results,
    "Katz Back-off": katz_results,
    "Stupid Back-off": stupid_results,
    "Kneser-Ney":    kn_results,
}

def fmt(val):
    return f"{val:,.2f}" if not math.isinf(val) else "∞"

# ── Development Set ──────────────────────────────────────────────────────
print("=" * 100)
print("DEVELOPMENT SET PERPLEXITY")
print("=" * 100)
print()
print(f"{'Method':<18} {'Unigram':>14} {'Bigram':>14} {'Trigram':>14} {'Quadgram':>14}")
print("-" * 78)
for method, results in all_methods.items():
    vals = [fmt(results[n]["dev"]) for n in ORDERS]
    print(f"{method:<18} {vals[0]:>14} {vals[1]:>14} {vals[2]:>14} {vals[3]:>14}")

# ── Test Set ─────────────────────────────────────────────────────────────
print()
print("=" * 100)
print("TEST SET PERPLEXITY")
print("=" * 100)
print()
print(f"{'Method':<18} {'Unigram':>14} {'Bigram':>14} {'Trigram':>14} {'Quadgram':>14}")
print("-" * 78)
for method, results in all_methods.items():
    vals = [fmt(results[n]["test"]) for n in ORDERS]
    print(f"{method:<18} {vals[0]:>14} {vals[1]:>14} {vals[2]:>14} {vals[3]:>14}")

---
## 13. Probability Sanity Checks & Sample Sentence Analysis

In [ ]:
# ── Probability sanity checks ────────────────────────────────────────────
methods_fns = {
    "Interpolated":    interpolated_prob,
    "Good-Turing":     good_turing_prob,
    "Katz Back-off":   katz_prob,
    "Stupid Back-off": stupid_backoff_prob,
    "Kneser-Ney":      kneser_ney_prob,
}

check_examples = [
    (2, (SOS,),           'ఈ'),
    (2, ('.',),           EOS),
    (3, (SOS, SOS),       'ఈ'),
    (4, (SOS, SOS, SOS),  'ఈ'),
]

print("Probability Sanity Checks")
print("=" * 80)

for order, history, word in check_examples:
    hist_str = ", ".join(history) if history else "(none)"
    print(f"\nP({word} | {hist_str})  [{ORDER_NAMES[order]}]")
    for method_name, prob_fn in methods_fns.items():
        p = prob_fn(order, history, word)
        print(f"  {method_name:<18}: {p:.10f}")

In [ ]:
# ── Sample sentence log-probabilities ────────────────────────────────────
sample = test_sentences[0]
print("Sample test sentence:")
print(" ".join(sample[:20]), "..." if len(sample) > 20 else "")
print()

for method_name, prob_fn in methods_fns.items():
    print(f"{method_name}:")
    for n in ORDERS:
        lp, nt = sentence_log_prob(sample, prob_fn, n)
        print(f"  {ORDER_NAMES[n]:<12} log₂P = {lp:>12.4f}  (tokens = {nt})")
    print()

---
## 14. Summary & Observations

In [ ]:
print("=" * 100)
print("SUMMARY — N-Gram Smoothing Techniques (Assignment 6)")
print("=" * 100)
print()
print(f"Corpus           : IndicCorpV2 Telugu (tel_Telu)")
print(f"Total sentences  : {NUM_SENTENCES:,}")
print(f"Training set     : {TRAIN_SIZE:,} sentences")
print(f"Development set  : {DEV_SIZE:,} sentences")
print(f"Test set         : {TEST_SIZE:,} sentences")
print(f"Vocabulary size  : {V:,}")
print()
print("Smoothing techniques implemented:")
print("  1. Interpolated Smoothing   — weighted combination of MLE from all sub-orders")
print(f"     Weights: Bigram={INTERP_WEIGHTS[2]}, Trigram={INTERP_WEIGHTS[3]}, Quadgram={INTERP_WEIGHTS[4]}")
print("  2. Good-Turing Smoothing    — adjusts counts using frequency-of-frequencies")
print("  3. Katz Back-off Smoothing  — GT discounting for seen n-grams, back-off for unseen")
print(f"  4. Stupid Back-off          — α={STUPID_ALPHA}, simple back-off (not a true distribution)")
print(f"  5. Kneser-Ney Smoothing     — d={KN_DISCOUNT}, absolute discounting + continuation probability")
print()

# ── Find best method for each model order on dev set ─────────────────────
print("Best method per model (lowest Dev Perplexity):")
print("-" * 60)
for n in ORDERS:
    best_method = min(all_methods.keys(), key=lambda m: all_methods[m][n]["dev"])
    best_pp = all_methods[best_method][n]["dev"]
    print(f"  {ORDER_NAMES[n]:<12} → {best_method:<18} (PP = {best_pp:,.2f})")

print()
print("Key observations:")
print("  - Kneser-Ney typically achieves the best perplexity due to its use of")
print("    continuation probabilities that better model word usage diversity.")
print("  - Good-Turing and Katz Back-off provide solid improvements over MLE")
print("    by redistributing probability mass from seen to unseen events.")
print("  - Stupid Back-off is computationally simple and effective for large corpora,")
print("    though its scores are not true probabilities.")
print("  - Interpolation benefits from combining evidence across all model orders,")
print("    but depends on well-tuned weights (ideally via EM on held-out data).")
print("  - All methods eliminate the infinite perplexity problem of raw MLE.")
print()
print("=" * 100)